<a href="https://colab.research.google.com/github/rodfmattos/AZ-104-MicrosoftAzureAdministrator/blob/master/HandsOn_01_02_Mackenzie_MLXP_Rodrigo_F_Mattos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Célula 1: Configuração e Importação ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações para os gráficos
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Bibliotecas importadas com sucesso!")

In [ ]:
# --- Célula 2: Carregando os Dados ---

# O dataset C-MAPSS (FD001) não tem cabeçalho.
# Vamos carregar o dataset de treino (train_FD001.txt) de um repositório público.
# Se você tiver o arquivo, pode fazer upload e usar pd.read_csv("train_FD001.txt", ...)

# URL para os dados de treino (FD001)
url_train = 'https://raw.githubusercontent.com/microsoft/R-server-RUL-predict/master/Data/train_FD001.txt'

# Definindo os nomes das colunas (conforme documentação da NASA)
column_names = [
    'unit_number', 'time_in_cycles',
    'setting_1', 'setting_2', 'setting_3',
    'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5',
    'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10',
    'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15',
    'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21'
]

# Carregar os dados
# O arquivo é separado por espaços e não tem cabeçalho.
try:
    df_train = pd.read_csv(url_train, sep=' ', header=None)

    # O carregamento pode criar colunas extras vazias (NaN) no final. Vamos removê-las.
    df_train.drop(columns=[26, 27], inplace=True, errors='ignore')

    # Atribuir os nomes das colunas
    df_train.columns = column_names

    print("Dados de treino (FD001) carregados com sucesso.")
    print(f"Formato do dataset: {df_train.shape}")
    display(df_train.head())

except Exception as e:
    print(f"Erro ao carregar os dados: {e}")
    print("Por favor, verifique a URL ou faça upload do arquivo 'train_FD001.txt' manualmente.")

In [ ]:
# --- Célula 3: EDA e Resumo de Dados (Seus pontos 1 e 2) ---

print("--- Informações Gerais do Dataset ---")
df_train.info()

print("\n\n--- Resumo Estatístico (Medidas de Dados) ---")
# .transpose() (ou .T) facilita a leitura quando há muitas colunas
display(df_train.describe().transpose())

# Análise inicial:
# - Quantos motores (unidades) temos?
total_motores = df_train['unit_number'].nunique()
print(f"\nTotal de motores no dataset: {total_motores}")

# - Qual a distribuição do tempo de vida (ciclos)?
print("\n--- Distribuição do Tempo de Vida (Ciclos) ---")
display(df_train.groupby('unit_number')['time_in_cycles'].max().describe())

In [ ]:
# --- Célula 4: Pré-processamento e Análise Bivariada (Seu ponto 3) ---

# 1. Identificar e remover sensores "inúteis" (constantes)
# Um sensor "constante" terá um desvio padrão (std) de 0.

print("--- Identificando Sensores Constantes (Desvio Padrão = 0) ---")
sensor_std = df_train.std()
sensores_constantes = sensor_std[sensor_std == 0].index.tolist()

if sensores_constantes:
    print(f"Sensores constantes identificados: {sensores_constantes}")
    # Remover esses sensores do dataframe
    df_train = df_train.drop(columns=sensores_constantes)
    print("Sensores constantes removidos.")
else:
    print("Nenhum sensor constante encontrado.")


# 2. Engenharia de Atributos: Criar o "Remaining Useful Life" (RUL)
# O RUL é a nossa variável-alvo (target).
# RUL = (Ciclo máximo daquele motor) - (Ciclo atual)

# Encontrar o ciclo máximo para cada motor
max_cycles = df_train.groupby('unit_number')['time_in_cycles'].max().reset_index()
max_cycles.columns = ['unit_number', 'max_cycles']

# Juntar (merge) essa informação de volta ao dataframe principal
df_train = pd.merge(df_train, max_cycles, on='unit_number', how='left')

# Calcular o RUL
df_train['RUL'] = df_train['max_cycles'] - df_train['time_in_cycles']

# Agora podemos dropar a coluna 'max_cycles' que era auxiliar
df_train = df_train.drop(columns=['max_cycles'])

print("\n--- Coluna 'RUL' criada com sucesso! ---")
display(df_train[['unit_number', 'time_in_cycles', 'RUL']].head())


# 3. Análise Bivariada: Matriz de Correlação
# Agora que temos o RUL, podemos ver quais sensores se correlacionam com ele.
print("\n--- Correlação das Variáveis com o RUL ---")
corr_matrix = df_train.corr()
rul_correlation = corr_matrix['RUL'].sort_values(ascending=False)

display(rul_correlation)

In [ ]:
# --- Célula 5: Visualização de Dados (Seu ponto 4) ---

# 1. Visualização: Heatmap de Correlação
# Isso nos dá a visão geral de (Seu ponto 3)
plt.figure(figsize=(20, 15))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            cbar_kws={'label': 'Coeficiente de Correlação'})
plt.title('Heatmap de Correlação de todas as Variáveis (Incluindo RUL)', fontsize=16)
plt.show()

# 2. Visualização: A "Curva da Degradação" (Seu ponto 4)
# Vamos plotar os sensores com maior correlação (negativa ou positiva) com o RUL.
# Do passo anterior, vimos que 'sensor_11', 'sensor_4', 'sensor_7', 'sensor_12' são fortes candidatos.

# Vamos plotar o sensor 11 ao longo do tempo para alguns motores
print("\n--- Visualizando a Degradação (Sensor vs. Tempo) ---")
plt.figure(figsize=(14, 7))

# Pegar uma amostra de 5 motores para não poluir o gráfico
motores_amostra = df_train[df_train['unit_number'].isin([1, 2, 3, 4, 5])]

sns.lineplot(data=motores_amostra, x='time_in_cycles', y='sensor_11', hue='unit_number', palette='tab10')
plt.title('Curva de Degradação: Sensor 11', fontsize=16)
plt.xlabel('Tempo em Ciclos')
plt.ylabel('Leitura do Sensor 11')
plt.legend(title='Motor (Unit)')
plt.show()

# 3. Visualização: Relação Direta (Sensor vs. RUL)
# Um scatter plot mostrando como o sensor se comporta à medida que o RUL diminui
plt.figure(figsize=(14, 7))
sns.scatterplot(data=motores_amostra, x='RUL', y='sensor_11', hue='unit_number', palette='tab10', alpha=0.6)
plt.title('Relação Direta: RUL vs. Sensor 11', fontsize=16)
plt.xlabel('Vida Útil Remanescente (RUL)')
plt.ylabel('Leitura do Sensor 11')
plt.gca().invert_xaxis() # Inverter o eixo X para ler "do mais saudável para a falha"
plt.show()

In [ ]:
# --- Célula 6: Storytelling de Dados (Seu ponto 5) ---

## 📖 A História da Falha: O que os Dados nos Contam

### 1. O Desafio: A Caixa Preta
Iniciamos com dados de 100 motores (`total_motores = 100`) sem saber por que ou quando eles falham. Vimos que o tempo de vida varia muito: alguns motores falham em 128 ciclos, enquanto outros duram até 362 (como visto na Célula 3). O desafio é encontrar um "sinal" universal de falha.

### 2. A Investigação: Separando o Sinal do Ruído
Nossa primeira análise estatística (Célula 3 e 4) foi crucial. Descobrimos que vários sensores (como `sensor_1`, `sensor_5`, `sensor_10`, etc.) são **totalmente inúteis** — eles nunca mudam (desvio padrão zero) e foram removidos. Isso limpou nosso dataset, deixando apenas sensores que de fato registram mudanças.

### 3. A Descoberta: A "Impressão Digital" da Falha
Ao criar a variável-alvo (o **RUL**) e cruzá-la com os sensores restantes, o padrão emergiu (Célula 4). O *Heatmap de Correlação* (Célula 5) foi o nosso "mapa do tesouro":

* **Sensores-Chave:** `sensor_11`, `sensor_4`, `sensor_7`, `sensor_12` e `sensor_15` mostraram uma **fortíssima correlação** com o RUL.
* **O que isso significa:** À medida que o motor se degrada (RUL diminui), esses sensores mudam de forma consistente e previsível.

### 4. A Prova: Visualizando a Degradação
O gráfico da "Curva de Degradação" (Célula 5) provou a hipótese. Vimos que o `sensor_11`, por exemplo, opera de forma estável (apenas "ruído") durante a maior parte da vida do motor, mas **começa a subir drasticamente** nos últimos ~100 ciclos antes da falha.



O *Scatter Plot* (RUL vs. Sensor) confirmou: quando o RUL é alto (motor saudável), as leituras são baixas. Quando o RUL se aproxima de zero (falha iminente), as leituras disparam.

### 5. A Solução: De Reativo para Preditivo
A história está clara: a falha do motor não é silenciosa. Ela "grita" através de sensores específicos.

Não precisamos mais adivinhar. Os dados mostram que podemos treinar um modelo de Machine Learning (como uma Regressão, Random Forest ou uma Rede Neural LSTM) usando esses sensores-chave como *features* para prever o *RUL* como *target*.

**O resultado:** Podemos parar de consertar motores que estão bons (manutenção preventiva) e parar de sermos pegos de surpresa (manutenção reativa). Estamos prontos para implementar um sistema que nos diz, com base nesses sensores, "Este motor precisa de manutenção nos próximos X ciclos".

In [ ]:
# --- Célula 7: Preparação para o Modelo (Divisão Treino/Teste) ---
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

print("Bibliotecas de Machine Learning importadas.")

# 1. Definir Features (X) e Alvo (y)
# Vamos usar apenas os sensores e configurações, não o RUL ou dados de ID
# Primeiro, identificar os sensores que sobraram
sensores_e_settings = [col for col in df_train.columns if 'sensor_' in col or 'setting_' in col]

X = df_train[sensores_e_settings]
y = df_train['RUL']

print(f"\nFeatures (X) selecionadas ({len(sensores_e_settings)}): {sensores_e_settings}")
print(f"Alvo (y): RUL")

# 2. Dividir os dados em Treino e Teste
# Usaremos uma divisão 80/20.
# O 'random_state' garante que a divisão seja sempre a mesma (reprodutibilidade)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nFormato dos dados de Treino (X_train): {X_train.shape}")
print(f"Formato dos dados de Teste (X_test): {X_test.shape}")